# Debug du modèle de détection de masque

Notebook de diagnostic séparé du notebook d'entraînement.

Objectifs :
- confirmer l'ordre des classes utilisé par l'entraînement ;
- exécuter une prédiction sur l'image fournie ;
- sauvegarder l'image prétraitée dans le dossier temporaire système pour inspection sans encombrer le dépôt.

In [ ]:
from pathlib import Path
import json
import tempfile
import numpy as np
from PIL import Image
import tensorflow as tf

PROJECT_ROOT = Path(r"C:/Users/yassi/OneDrive/Bureau/Portfolio/plugins/mask-detection-cnn")
MODEL_PATH = PROJECT_ROOT / 'models' / 'mask_detection_model.keras'
IMAGE_PATH = Path(r"C:/Users/yassi/OneDrive/Bureau/with_mask_1017.jpg")
CLASS_NAMES = ('with_mask', 'without_mask')
IMAGE_SIZE = 128

print('MODEL_PATH =', MODEL_PATH)
print('IMAGE_PATH =', IMAGE_PATH)
print('CLASS_NAMES =', CLASS_NAMES)

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Image not found: {IMAGE_PATH}')

model = tf.keras.models.load_model(str(MODEL_PATH))
print('Model output shape:', model.output_shape)

img = Image.open(IMAGE_PATH).convert('RGB')
w, h = img.size
side = min(w, h)
left = (w - side) // 2
top = (h - side) // 2
crop = img.crop((left, top, left + side, top + side))
resized = crop.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
array = np.asarray(resized, dtype=np.float32) / 255.0
input_tensor = np.expand_dims(array, axis=0)

temp_dir = Path(tempfile.gettempdir()) / 'mask-detection-cnn' / 'preprocessed'
temp_dir.mkdir(parents=True, exist_ok=True)
preprocessed_path = temp_dir / 'debug_preprocessed_with_mask_1017.png'
Image.fromarray((array * 255).astype('uint8')).save(preprocessed_path)
print('Saved preprocessed image to:', preprocessed_path)

display(resized)

In [ ]:
prediction = model.predict(input_tensor, verbose=0)[0]
inverted = np.array([prediction[1], prediction[0]])
result = {
    'raw_prediction': [float(x) for x in prediction],
    'all_probabilities': {CLASS_NAMES[i]: float(inverted[i]) for i in range(len(CLASS_NAMES))},
    'label': CLASS_NAMES[int(np.argmax(inverted))],
    'confidence': float(np.max(inverted)),
    'preprocessed_image': str(preprocessed_path),
}
print(json.dumps(result, indent=2, ensure_ascii=False))